In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, f1_score, recall_score
import joblib

# 1. Load the data
df = pd.read_csv('churn.csv')

# 2. Preprocess
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

# 3. Encode Categorical Variables
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# 4. Select the 9 Most Important Features
features_to_keep = [
    'Contract', 'tenure', 'MonthlyCharges', 'OnlineSecurity',
    'TotalCharges', 'TechSupport', 'PaymentMethod',
    'PaperlessBilling', 'InternetService'
]

X = df[features_to_keep]
y = df['Churn']

# 5. Split and Scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test) # Scale the test set for evaluation

# 6. Train the XGBoost Model
ratio = float(y_train.value_counts()[0]) / y_train.value_counts()[1]

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=ratio,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

# 7. Evaluate the Model with the Optimal Threshold
probabilities = xgb_model.predict_proba(X_test)[:, 1]
best_threshold = 0.5857
optimal_predictions = np.where(probabilities >= best_threshold, 1, 0)

print(f"--- XGBoost Model Evaluation (Threshold: {best_threshold}) ---")
print(f"Accuracy: {accuracy_score(y_test, optimal_predictions):.4f}")
print(f"Recall (Class 1): {recall_score(y_test, optimal_predictions):.4f}")
print(f"F1 Score (Class 1): {f1_score(y_test, optimal_predictions):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, optimal_predictions))

# 8. Save the Model and Scaler for the FastAPI Backend
joblib.dump(xgb_model, 'xgboost_churn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("\nModel and Scaler successfully saved to disk.")

--- XGBoost Model Evaluation (Threshold: 0.5857) ---
Accuracy: 0.7669
Recall (Class 1): 0.6872
F1 Score (Class 1): 0.6105

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.80      0.83      1033
           1       0.55      0.69      0.61       374

    accuracy                           0.77      1407
   macro avg       0.71      0.74      0.72      1407
weighted avg       0.79      0.77      0.77      1407


Model and Scaler successfully saved to disk.
